In [4]:
!pip install pymupdf -q

In [5]:
import fitz  # pymupdf

def extract_text_from_pdf(pdf_path):
    text_blocks = []
    with fitz.open(pdf_path) as pdf_document:
        for page in pdf_document:
            text = page.get_text("text").strip()
            if text:
                text_blocks.append(text)
    return text_blocks

In [6]:
pdf_texts = extract_text_from_pdf('/content/Metformin.pdf')
pdf_texts

['Metformin is one of the most widely prescribed oral antihyperglycemic agents.\u200b\n Its primary mechanism of action involves the activation of AMP-activated protein kinase \n(AMPK), a central metabolic regulator that promotes glucose uptake and fatty acid oxidation \nwhile inhibiting hepatic gluconeogenesis.\u200b\n Beyond its glycemic control, Metformin has been shown to improve cardiovascular outcomes \nand display anti-inflammatory properties.\u200b\n Recent studies also suggest potential anticancer effects through inhibition of the mTOR \nsignaling pathway and suppression of tumor angiogenesis. \n \nClinical trials have demonstrated that combining Atorvastatin with Ezetimibe results in \nsignificant reductions in low-density lipoprotein cholesterol (LDL-C) levels compared to \nmonotherapy.\u200b\n Ezetimibe acts by inhibiting the Niemann–Pick C1-like 1 (NPC1L1) transporter in the intestinal \nwall, reducing cholesterol absorption, while Atorvastatin inhibits hepatic HMG-CoA red

In [7]:
!pip install datasets transformers trl peft accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 761.1/761.1 kB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 18.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 12.5 MB/s eta 0:00:00


In [8]:
!pip install -U transformers -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 71.9 MB/s eta 0:00:00


In [9]:
!transformers version

5.9.0


In [10]:
#dataset maker
import re
from datasets import Dataset, load_dataset
def make_dataset(pdf_texts):
    paragraphs = []
    for page_text in pdf_texts:
        chunks = re.split(r'\n\s*\n', page_text)
        for chunk in chunks:
            chunk = chunk.strip()
            if len(chunk) > 30:
                paragraphs.append(chunk)
    dataset = [{'text' : p} for p in paragraphs]
    dataset = Dataset.from_list(dataset)
    return dataset

In [11]:
dataset = make_dataset(pdf_texts)


In [12]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

In [13]:
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments, DataCollatorForLanguageModeling

In [14]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [15]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [16]:
def tokenize(example):
    tokens =tokenizer(example['text'], truncation = True, padding = True, max_length = 512)
    tokens['label'] = tokens['input_ids'].copy()
    return tokens

In [17]:
tokenized = dataset.map(tokenize, batched = True, remove_columns = ['text'])

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [18]:
model = AutoModelForCausalLM.from_pretrained(model_name)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

In [19]:
training_arg = TrainingArguments(
    output_dir = "./llama-pharma-new",
    num_train_epochs = 2,
    per_device_train_batch_size = 2,
    save_steps = 500,
    save_total_limit = 2,
    logging_steps = 50,
    save_strategy = "steps",
    fp16 = True,
    report_to = "none"
)

In [20]:
trainer = Trainer(
    model = model,
    args = training_arg,
    train_dataset = tokenized,
    data_collator = DataCollatorForLanguageModeling(tokenizer, mlm = False)
)

In [39]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [23]:
!pip install -U bitsandbytes accelerate peft -q

In [24]:
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset

In [44]:
model = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def tokenize_fn(examples):
    tokens = tokenizer(
        examples["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized = dataset.map(tokenize_fn, batched=True)

Map:   0%|          | 0/4 [00:00<?, ? examples/s]

In [46]:
from transformers import BitsAndBytesConfig
import torch

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=False,
)

model = AutoModelForCausalLM.from_pretrained(
    model,
    device_map="auto",
    quantization_config=quantization_config
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/bitsandbytes/backends/default/ops.py:223: FutureWarning: _check_is_size will be removed in a future PyTorch release along with guard_size_oblivious.     Use _check(i >= 0) instead.
  torch._check_is_size(blocksize)


In [47]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj","v_proj"],
    lora_dropout=0.05,
    bias="none"
)

In [36]:
!pip install -U torchao

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 48.3 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [51]:
# save adapter
non_inst_model_lora = get_peft_model(model, lora_config)
non_inst_model_lora.config.use_cache = False # Disable cache for memory efficiency during training

#args
args = TrainingArguments(
    output_dir="./tinyllama-lora",
    num_train_epochs=5,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none",
    gradient_checkpointing=True # Enable gradient checkpointing to save memory
)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [52]:
trainer = Trainer(model = non_inst_model_lora , args = args, train_dataset = tokenized)

In [53]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=5, training_loss=10.004911804199219, metrics={'train_runtime': 11.434, 'train_samples_per_second': 1.749, 'train_steps_per_second': 0.437, 'total_flos': 63629646888960.0, 'train_loss': 10.004911804199219, 'epoch': 5.0})

In [54]:
model_path = "/content/tinyllama-lora/checkpoint-5"

# tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForCausalLM.from_pretrained(model_path, device_map="auto")

prompt = "Clinical trials demonstrated that combining Atorvastatin with Ezetimibe"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

outputs = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.8,
    top_p=0.9,
    do_sample=True,
    repetition_penalty=1.1
)

print("\nModel Output:\n")
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/88 [00:00<?, ?it/s]

[transformers] Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



Model Output:

Clinical trials demonstrated that combining Atorvastatin with Ezetimibe was more effective than Atorvastatin and placebo in reducing LDL-C. In addition, the data suggest that Ezetimibe is associated with less adverse events than Atorvastatin.
Pravastatinum is a combination of pravastatin (10 mg) with simvastatin (20 mg). Both statins have been used as first line therapy for cholesterol lowering in patients with hypercholest
